# Agentic RAG: Router-Retriever System with PDF and Web Search Tools

**Overview**
This project challenges you to design and implement an Agentic Retrieval-Augmented Generation (RAG) system that employs multiple role-based AI agents to process user queries. The workflow includes a Router Agent that classifies questions and a Retriever Agent that executes the most suitable retrieval method, using either a PDF-based vector search or a web search tool. The system demonstrates intelligent orchestration across these agents, producing accurate, source-grounded answers.

**Instructions**
- Review the learning modules and any linked documentation for CrewAI and the tools used
- Read each section (situation, tasks, actions, and result) to understand the deliverables and workflow expectations thoroughly
- Implement the complete solution using Python in Jupyter notebook, with appropriate orchestration and logging tools
- Submit your final working notebook and a brief README file through the LMS
- Ensure all project elements, from role definitions to reasoning trace visualizations, are included and functional

**Situation**
As a developer, you are tasked with building an AI-powered multi-agent system to answer domain-specific questions using both static and dynamic sources. The system comprises:
- Router agent: It decides the retrieval path based on the question
- Retriever agent: It executes the retrieval from the chosen source and formulates the answer
Available tools include a PDF search tool for static, domain-specific content and a Web search tool for retrieving fresh information from the internet. An optional generation path directly uses the LLM without retrieval.

In [ ]:
%pip install langchain langchain-openai langchain-community 
%pip install langchain-text-splitters langchain-classic langchain-core faiss-cpu pypdf
%pip install langchain-tavily tavily-python 
%pip install crewai
%pip install litellm
%pip install pydantic dotenv

In [ ]:
# Import environment variables
import os
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

# Check if API keys were loaded from environment variables
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable not found!")
    
if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY environment variable not found!")


In [ ]:
# Build web search tool using tavily
import requests
from crewai.tools import BaseTool

class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information"

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": os.getenv("TAVILY_API_KEY"),
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload)
        data = response.json()
        
        results = []
        for r in data["results"]:
                results.append(f"{r['title']} - {r['url']}")
        
        return "\n".join(results)

search_tool = TavilySearchTool()

In [ ]:
# Split the PDF and create a vector store for retrieval
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# Load PDF document
loader = PyPDFLoader(
    file_path="input/AI-interview-questions-2026.pdf",
    extraction_mode="layout"
)
pdf_document=loader.load()
print(f"Loaded {len(pdf_document)} pages")

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50
)
pdf_chunks=text_splitter.split_documents(pdf_document)
print(f"Split into {len(pdf_chunks)} chunks")

# Generate embeddings for the chunks
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002",
    api_key=os.getenv("OPENAI_API_KEY")
)
vectorstore = FAISS.from_documents(pdf_chunks, embeddings)

In [ ]:
# Build PDF RAG search tool
# Returns raw retrieved snippets (like TavilySearchTool returns raw titles/URLs)
# so the Retriever agent formulates the final answer itself, keeping its
# reasoning visible in the trace instead of hiding it behind a second LLM call.

from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

class PDFSearchTool(BaseTool):
    name: str = "PDF Search"
    description: str = "Search the AI interview questions PDF for domain-specific reference content."

    def _run(self, query: str):
        docs = vectorstore.similarity_search(query, k=3)
        results = []
        for d in docs:
            page = d.metadata.get("page", "unknown")
            snippet = d.page_content.strip().replace("\n", " ")[:500]
            results.append(f"[Page {page}] {snippet}")
        return "\n\n".join(results)

pdf_tool = PDFSearchTool()

## Router-Retriever coordination

A hierarchical CrewAI process is used: the **Router** agent is the crew's
`manager_agent` and classifies each question as `PDF`, `WEB`, or `DIRECT`,
then delegates to the **Retriever** agent (which holds both tools) with
instructions on which single tool to use. Assigning the one `Task` to
`retriever` restricts the Router's delegation target to exactly that agent,
matching the assignment's Router → Retriever shape.

In [ ]:
from crewai.llm import LLM
from crewai import Agent, Task, Crew, Process

router_llm = LLM(
    model=os.getenv("OPENAI_MODEL_NAME"),
    api_key=os.getenv("OPENAI_API_KEY"),
    is_litellm=True,
    temperature=0
)
retriever_llm = LLM(
    model=os.getenv("OPENAI_MODEL_NAME"),
    api_key=os.getenv("OPENAI_API_KEY"),
    is_litellm=True,
    temperature=0.3
)

def log_task(output):
    print(f"\n--- TASK COMPLETE ---\n{output.raw}\n")

async def run_agentic_rag(user_question: str):
    router = Agent(
        role="Query Router",
        goal="Classify the question as PDF, WEB, or DIRECT, then delegate it to the Answer Retriever "
             "with clear instructions on which single tool (if any) to use.",
        backstory=("A triage analyst who knows the AI-interview-questions-2026.pdf reference document's "
                   "scope and decides whether a question is answerable from it, needs fresh web info, "
                   "or needs no retrieval at all."),
        llm=router_llm,
        verbose=True,
        allow_delegation=True,
    )

    retriever = Agent(
        role="Answer Retriever",
        goal="Follow the Router's instructions, use at most one retrieval tool if told to, "
             "and formulate a grounded final answer.",
        backstory=("A careful research assistant that only uses the source it's instructed to use "
                   "and never fabricates facts beyond what it retrieves."),
        llm=retriever_llm,
        tools=[pdf_tool, search_tool],
        verbose=True,
    )

    task_answer = Task(
        description=(
            f"Answer this question: {user_question}\n\n"
            "First classify it as PDF (answerable from the static AI-interview-questions-2026.pdf "
            "reference), WEB (needs fresh/current info), or DIRECT (general knowledge, no retrieval). "
            "Then delegate to the Answer Retriever, telling it exactly which single tool to use "
            "(PDF Search, Web Search, or none) and have it produce the final answer."
        ),
        expected_output="A clear final answer that states which source (PDF, web, or general knowledge) was used.",
        agent=retriever,
        callback=log_task,
    )

    crew = Crew(
        agents=[retriever],          # manager_agent must NOT be in this list
        tasks=[task_answer],
        process=Process.hierarchical,
        manager_agent=router,
        verbose=True,
        output_log_file="trace.log",
        tracing=True,
    )

    # Jupyter runs its own asyncio event loop, so a synchronous crew.kickoff()
    # raises "invoked synchronously from within a running event loop" here.
    result = await crew.kickoff_async()
    print(f"\n=== Question: {user_question} ===\n{result.raw}\n")
    return {
        "question": user_question,
        "answer": result.raw,
        "crew_token_usage": result.token_usage,
        "router_token_usage": router_llm.get_token_usage_summary(),
        "retriever_token_usage": retriever_llm.get_token_usage_summary(),
    }

In [ ]:
# Run sample questions covering all three routing paths
sample_questions = [
    "What are common AI interview questions about overfitting?",  # PDF
    "What AI hiring trends are being discussed in 2026?",         # WEB
    "What is 2 + 2?",                                             # DIRECT
]

runs = []
for q in sample_questions:
    runs.append(await run_agentic_rag(q))

In [ ]:
# Summary table tying question -> answer -> token cost together
for run in runs:
    print(f"Question: {run['question']}")
    print(f"Answer: {run['answer'][:200]}")
    print(f"Crew tokens: {run['crew_token_usage']}")
    print(f"Router tokens: {run['router_token_usage']}")
    print(f"Retriever tokens: {run['retriever_token_usage']}")
    print("-" * 80)